In [1]:
# ============================================================================
# SECTION 1: Import Required Libraries
# ============================================================================

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.io import wavfile
import librosa
import librosa.display
from scipy.signal import lfilter
from scipy.signal.windows import hamming
import warnings
warnings.filterwarnings('ignore')

print("All libraries imported successfully!")

All libraries imported successfully!


In [2]:
# ============================================================================
# SECTION 3: LPC Coefficient Calculation Functions
# ============================================================================

def autocorr(x, lag=20):
    """
    Calculate autocorrelation using FFT method
    """
    n = len(x)
    x = x - np.mean(x)
    r = np.correlate(x, x, mode='full')[-n:]
    result = r / (n * np.var(x))
    return result[:lag+1]

def levinson_durbin(r, order):
    """
    Levinson-Durbin recursion for LPC coefficient calculation
    """
    a = np.zeros(order + 1)
    e = np.zeros(order + 1)
    a[0] = 1.0
    e[0] = r[0]
    
    for i in range(1, order + 1):
        lambda_val = sum(a[j] * r[i-j] for j in range(i))
        k = -lambda_val / e[i-1]
        
        a_new = np.zeros(order + 1)
        a_new[0] = 1.0
        for j in range(1, i):
            a_new[j] = a[j] + k * a[i-j]
        a_new[i] = k
        a = a_new
        e[i] = (1 - k**2) * e[i-1]
    
    return a, e[-1]

def lpc_analysis(frame, order=12):
    """
    Perform LPC analysis on a speech frame
    Returns LPC coefficients and prediction error
    """
    # Apply Hamming window
    windowed = frame * hamming(len(frame))
    
    # Calculate autocorrelation
    r = autocorr(windowed, order)
    
    # Apply Levinson-Durbin algorithm
    lpc_coeffs, error = levinson_durbin(r, order)
    
    return lpc_coeffs, error

print("\nLPC Analysis Functions defined successfully!")


LPC Analysis Functions defined successfully!


In [3]:
# ============================================================================
# SECTION 4: Speech Signal Acquisition
# ============================================================================

def create_synthetic_speech(duration=3, sr=16000):
    """
    Create a synthetic speech-like signal with vowel sounds
    """
    t = np.linspace(0, duration, int(sr * duration))
    speech = np.zeros_like(t)
    
    # Create different vowel segments
    vowels = [
        (240, 2400),   # /i/
        (750, 940),    # /ɑ/
        (390, 2300),   # /e/
        (250, 595),    # /u/
    ]
    
    segment_length = len(t) // len(vowels)
    
    for idx, (f1, f2) in enumerate(vowels):
        start = idx * segment_length
        end = (idx + 1) * segment_length if idx < len(vowels) - 1 else len(t)
        t_segment = t[start:end]
        
        # Generate glottal pulse train
        f0 = 120  # Fundamental frequency
        glottal = signal.square(2 * np.pi * f0 * t_segment)
        
        # Create formant filters with normalized frequencies (0 to 1)
        nyquist = sr / 2
        f1_norm = np.clip(f1 / nyquist, 0.01, 0.99)
        f2_norm = np.clip(f2 / nyquist, 0.01, 0.99)
        
        # Ensure f1 < f2 for bandpass filter
        if f1_norm > f2_norm:
            f1_norm, f2_norm = f2_norm, f1_norm
        
        b1, a1 = signal.butter(2, [f1_norm, f2_norm], btype='band', analog=False)
        b2, a2 = signal.butter(2, [f1_norm, f2_norm], btype='band', analog=False)
        
        # Apply formant filtering
        vowel_sound = signal.lfilter(b1, a1, glottal)
        vowel_sound = signal.lfilter(b2, a2, vowel_sound)
        
        speech[start:end] = vowel_sound
    
    # Normalize
    speech = speech / np.max(np.abs(speech)) * 0.8
    
    return speech, sr

# Generate synthetic speech
print("\nGenerating synthetic speech signal...")
speech_signal, sample_rate = create_synthetic_speech(duration=3, sr=16000)
print(f"Speech signal generated: {len(speech_signal)} samples at {sample_rate} Hz")
print(f"Duration: {len(speech_signal)/sample_rate:.2f} seconds")


Generating synthetic speech signal...
Speech signal generated: 48000 samples at 16000 Hz
Duration: 3.00 seconds
